# Freyra â€” Influencer Edition (T4 Colab)
Generate photorealistic influencer images on a free T4 GPU.
No technical knobs - just pick a shoot type, character, and pose.

Run **Cell 2** (setup), **Cell 3** (verify GPU), then **Cell 4** (launch).


In [ ]:
# â”€â”€ Cell 2 Â· Setup â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
%cd /content
!git clone https://github.com/Ashutosh94g/Freyra.git 2>/dev/null || git -C Freyra pull --ff-only
%cd /content/Freyra

# Remove cupy if present â€” it forces numpy 2.x which breaks the pipeline
!pip uninstall -yq cupy-cuda12x cupy-cuda11x cupy 2>/dev/null; true

# PyTorch 2.4.1 + CUDA 12.1 (verified on T4)
%pip install -q torch==2.4.1 torchvision==0.19.1 --extra-index-url https://download.pytorch.org/whl/cu121

# All project requirements (gradio 4.44.1, transformers>=4.45, numpy<2.0, etc.)
%pip install -q -r requirements_versions.txt

# Face swap dependencies (CPU-only onnxruntime to save VRAM)
%pip install -q insightface onnxruntime

# Cloudflared tunnel (backup when Gradio share is down)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# Create directories for Freyra v3 features
!mkdir -p characters assets/poses assets/thumbnails models/insightface

print("\nâœ… Setup complete â€” run Cell 3 to launch.")


In [ ]:
# â”€â”€ Cell 3 Â· Verify GPU & VRAM â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / 1024**3
    free_gb  = (props.total_memory - torch.cuda.memory_reserved(0)) / 1024**3
    print(f'GPU  : {props.name}')
    print(f'VRAM : {total_gb:.1f} GB total  |  {free_gb:.1f} GB free')
    if total_gb < 12:
        print('âš ï¸  Less than 12 GB VRAM â€” lower resolution to 832Ã—1152 in the UI.')
    else:
        print('âœ… VRAM looks good for influencer preset.')
else:
    print('âŒ No CUDA GPU â€” go to Runtime â†’ Change runtime type â†’ T4 GPU.')


In [ ]:
# â”€â”€ Cell 4 Â· Launch â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
%cd /content/Freyra
!python launch.py \
  --share \
  --preset influencer \
  --always-gpu \
  --unet-in-fp8-e4m3fn \
  --vae-in-fp16 \
  --attention-pytorch \
  --disable-image-log \
  --tunnel both


## Tips & Troubleshooting

| Problem | Fix |
|---|---|
| **OOM / CUDA out of memory** | Lower resolution to `832Ã—1152`, or disable 1â€“2 LoRAs in the UI (slots 3â€“4 are safest to disable) |
| **`ModuleNotFoundError: numpy`** | Re-run Cell 2 |
| **Gradio URL not showing** | Wait 60â€“90 s â€” checkpoint download is ~6.6 GB |
| **Black / broken images** | Try a different Shoot Type or reduce quality to Standard |

### T4 VRAM budget (fp8 UNet, 4 LoRAs, --always-gpu — single model)
| Component | VRAM |
|---|---|
| Base UNet (fp8) | ~5 GB |
| CLIP encoders | ~1.5 GB |
| VAE | ~0.3 GB |
| 4 LoRA patches | ~1.0 GB |
| Inference workspace | ~3.5 GB reserved |
| **Total** | **~11.3 GB / 15 GB** |

### T4 resolution guide
| Aspect | Resolution | Notes |
|---|---|---|
| Portrait 3:4 | `896Ã—1152` | Default â€” fits with all 4 LoRAs |
| Square 1:1 | `1024Ã—1024` | Fine |
| Landscape 4:3 | `1152Ã—896` | Fine |
| OOM fallback | `832Ã—1152` | Also disable LoRA slots 3â€“4 |
